6402-010302D Черных Вероника

Подключаем необходимые библиотеки

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [5]:
# Создание сесcии Spark
def create_session(app_name):
    return SparkSession.builder\
    .appName(app_name)\
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .getOrCreate()

In [6]:
# Чтение файлов формата CSV
def read_csv(spark, file_path):
    try:
        df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
        return df
    except Exception as e:
        print(f"Error: {file_path}: {e}")
        return None

In [7]:
# Функция для преобразования в числовой тип столбцов
def cast_columns_to_double(df, columns):
    for column in columns:
        df = df.withColumn(column, F.col(column).cast("double"))
    return df

In [8]:
# Сессия Spark
spark = create_session("BikeAnalysis")

In [12]:
# Чтение CSV-файлов
trips_df = read_csv(spark, "/trip.csv")
stations_df = read_csv(spark, "/station.csv")

In [13]:
# Приведение нужных колонок к числовому типу
columns_to_cast = ["lat", "long"]
stations_df = cast_columns_to_double(stations_df, columns_to_cast)

stations_df.show()
trips_df.show()

+---+--------------------+------------------+-------------------+----------+------------+-----------------+
| id|                name|               lat|               long|dock_count|        city|installation_date|
+---+--------------------+------------------+-------------------+----------+------------+-----------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|        27|    San Jose|         8/6/2013|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|        15|    San Jose|         8/5/2013|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|        11|    San Jose|         8/6/2013|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|        19|    San Jose|         8/5/2013|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|        15|    San Jose|         8/7/2013|
|  7|Paseo de San Antonio|         37.333798|-121.88694299999999|        15|    San Jose|         8/7/2013|
|  8| San Salvador at 1st|  

# Задания

1. Найти велосипед с максимальным временем пробега.

In [14]:
from pyspark.sql.functions import unix_timestamp, col, sum as spark_sum

# Сначала проверим, какие колонки есть в DataFrame
print("Доступные колонки в trips_df:", trips_df.columns)

# Проверим наличие нужных колонок
required_columns = ['start_date', 'end_date', 'bike_id']
if not all(col in trips_df.columns for col in required_columns):
    missing = [col for col in required_columns if col not in trips_df.columns]
    print(f"Ошибка: В данных отсутствуют необходимые колонки: {missing}")
    spark.stop()
    exit()

try:
    # Добавляем вычисляемые колонки только если нужные колонки существуют
    trips_with_duration = trips_df.withColumn(
        "start_timestamp", unix_timestamp(col("start_date"), "M/d/yyyy H:mm").cast("long")
    ).withColumn(
        "end_timestamp", unix_timestamp(col("end_date"), "M/d/yyyy H:mm").cast("long")
    ).withColumn(
        "duration_minutes", (col("end_timestamp") - col("start_timestamp")) / 60
    )

    # Группируем по bike_id и находим велосипед с максимальным временем пробега
    bike_max = trips_with_duration.groupBy("bike_id").agg(
        spark_sum("duration_minutes").alias("total_minutes")
    ).orderBy(col("total_minutes").desc()).limit(1)

    bike_max.show()

except Exception as e:
    print(f"Ошибка: {e}")
    spark.stop()
    exit()

Доступные колонки в trips_df: ['id', 'duration', 'start_date', 'start_station_name', 'start_station_id', 'end_date', 'end_station_name', 'end_station_id', 'bike_id', 'subscription_type', 'zip_code']
+-------+-------------+
|bike_id|total_minutes|
+-------+-------------+
|    535|     310262.0|
+-------+-------------+



2. Найти наибольшее геодезическое расстояние между станциями.

In [17]:
from pyspark.sql.functions import max, radians, sin, cos, sqrt, atan2, col

# Проверяем наличие необходимых колонок в stations_df
required_columns = ['id', 'lat', 'long']
if not all(col in stations_df.columns for col in required_columns):
    missing = [col for col in required_columns if col not in stations_df.columns]
    print(f"Ошибка: В данных stations_df отсутствуют необходимые колонки: {missing}")
    spark.stop()
    exit()

try:
    # Создаем пары станций (исключая одинаковые пары и дубликаты)
    station_pairs = stations_df.alias("station1").crossJoin(
        stations_df.alias("station2")
    ).filter(
        col("station1.id") < col("station2.id")
    )

    # Функция для расчета расстояния по формуле гаверсинуса
    def haversine(lat1, lon1, lat2, lon2):
        """Вычисляет расстояние между двумя точками на сфере (Земле)"""
        R = 6371.0  # Радиус Земли в километрах

        # Преобразуем координаты в радианы
        dlat = radians(lat2) - radians(lat1)
        dlon = radians(lon2) - radians(lon1)

        # Формула гаверсинуса
        a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
        c = 2 * atan2(sqrt(a), sqrt(1 - a))

        distance = R * c
        return distance

    # Регистрируем UDF функцию
    spark.udf.register("haversine", haversine)

    # Добавляем колонку с расстоянием
    station_pairs = station_pairs.withColumn(
        "distance",
        haversine(
            col("station1.lat"),
            col("station1.long"),
            col("station2.lat"),
            col("station2.long")
        )
    )

    # Проверяем, что расчет расстояния прошел успешно
    if "distance" not in station_pairs.columns:
        raise ValueError("Не удалось вычислить расстояние между станциями")

    # Вычисляем максимальное расстояние между станциями
    max_distance = station_pairs.agg(max("distance")).collect()[0][0]

    print(f"Максимальное геодезическое расстояние: {max_distance:.2f} км")

except Exception as e:
    print(f"Ошибка при расчете расстояний: {e}")
    spark.stop()
    exit()

Максимальное геодезическое расстояние: 69.92 км


3. Найти путь велосипеда с максимальным временем пробега через станции.

In [18]:
from pyspark.sql.functions import unix_timestamp, col, sum as spark_sum

# Проверяем наличие необходимых колонок в trips_df
required_columns = ['start_date', 'end_date', 'bike_id', 'start_station_name', 'end_station_name']
if not all(col_name in trips_df.columns for col_name in required_columns):
    missing = [col for col in required_columns if col not in trips_df.columns]
    print(f"Ошибка: В данных отсутствуют необходимые колонки: {missing}")
    exit()

try:
    # Преобразуем даты и вычисляем длительность поездок
    trips_with_duration = trips_df.withColumn(
        "start_timestamp", unix_timestamp(col("start_date"), "M/d/yyyy H:mm").cast("long")
    ).withColumn(
        "end_timestamp", unix_timestamp(col("end_date"), "M/d/yyyy H:mm").cast("long")
    ).withColumn(
        "duration_minutes", (col("end_timestamp") - col("start_timestamp")) / 60
    )
except Exception as e:
    print(f"Ошибка при преобразовании дат: {e}. Убедитесь, что формат даты соответствует 'M/d/yyyy H:mm'.")
    exit()

try:
    # Находим велосипед с максимальным временем пробега
    bike_max = trips_with_duration.groupBy("bike_id").agg(
        spark_sum("duration_minutes").alias("total_minutes")
    ).orderBy(col("total_minutes").desc()).limit(1)

    # Проверяем, что результат не пустой
    bike_max_list = bike_max.collect()
    if not bike_max_list:
        print("Не удалось найти велосипед с максимальным временем пробега.")
        exit()

    # Проверяем наличие bike_id в результате
    bike_id_max = bike_max_list[0]["bike_id"]

    # Получаем все поездки для этого велосипеда
    bike_trips = trips_with_duration.filter(col("bike_id") == bike_id_max) \
        .select("start_date", "start_station_name", "end_date", "end_station_name") \
        .orderBy("start_date")

    print(f"\nПоездки велосипеда {bike_id_max} с максимальным общим временем пробега:")
    bike_trips.show(truncate=False)

except Exception as e:
    print(f"Ошибка при анализе данных: {e}")
    exit()


Поездки велосипеда 535 с максимальным общим временем пробега:
+---------------+---------------------------------------------+---------------+---------------------------------------------+
|start_date     |start_station_name                           |end_date       |end_station_name                             |
+---------------+---------------------------------------------+---------------+---------------------------------------------+
|1/1/2014 13:42 |Mechanics Plaza (Market at Battery)          |1/1/2014 14:36 |Embarcadero at Sansome                       |
|1/1/2014 18:51 |Embarcadero at Sansome                       |1/1/2014 19:13 |Market at 4th                                |
|1/1/2014 19:48 |Market at 4th                                |1/1/2014 20:01 |South Van Ness at Market                     |
|1/10/2014 20:13|Market at 10th                               |1/10/2014 20:17|Powell Street BART                           |
|1/10/2014 8:09 |Embarcadero at Folsom                 


4. Найти количество велосипедов в системе.

In [21]:
bike_count = trips_df.groupBy("bike_id").agg(F.count("bike_id").alias("count")).count()

print(f"Количество велосипедов в системе: {bike_count} велосипед/ов")

Количество велосипедов в системе: 700 велосипед/ов


5. Найти пользователей потративших на поездки более 3 часов.

In [22]:
from pyspark.sql import functions as F

# Проверяем наличие необходимых колонок
required_columns = ['zip_code', 'duration_minutes']
if not all(col in trips_with_duration.columns for col in required_columns):
    missing = [col for col in required_columns if col not in trips_with_duration.columns]
    print(f"Ошибка: В данных отсутствуют необходимые колонки: {missing}")
    exit()

try:
    # Фильтруем пользователей с указанным почтовым индексом
    filtered_users = trips_with_duration.where(F.col("zip_code").isNotNull())

    # Проверяем, что после фильтрации остались данные
    if filtered_users.count() == 0:
        print("Нет данных о пользователях с указанными почтовыми индексами")
        exit()

    # Группируем данные по zip_code и считаем общее время поездок
    user_time = filtered_users.groupBy("zip_code").agg(
        F.sum("duration_minutes").alias("total_minutes")
    )

    # Фильтруем активных пользователей (> 180 минут)
    active_users = user_time.where(F.col("total_minutes") > 180)

    # Проверяем наличие активных пользователей
    if active_users.count() == 0:
        print("Нет пользователей, потративших более 180 минут на поездки")
    else:
        print("Активные пользователи (более 180 минут поездок):")
        active_users.select("zip_code", "total_minutes").orderBy(F.col("total_minutes").desc()).show(truncate=False)

except Exception as e:
    print(f"Ошибка при анализе данных: {e}")
    exit()

Активные пользователи (более 180 минут поездок):
+--------+-------------+
|zip_code|total_minutes|
+--------+-------------+
|94107   |830020.0     |
|nil     |762124.0     |
|94105   |426380.0     |
|94133   |360753.0     |
|94102   |318746.0     |
|94103   |318675.0     |
|95531   |287899.0     |
|94111   |237557.0     |
|95112   |212292.0     |
|94109   |200905.0     |
|94040   |130163.0     |
|94110   |123710.0     |
|94117   |115046.0     |
|94301   |109852.0     |
|94041   |104673.0     |
|94158   |104277.0     |
|94306   |92559.0      |
|94025   |86350.0      |
|94108   |85606.0      |
|94611   |83476.0      |
+--------+-------------+
only showing top 20 rows



In [23]:
# Остановка SparkSession
spark.stop()